# 06b — Per-class summary + Figure gallery (ไม่ใช้ GPU, รันได้ทันทีหลัง nb06)

1. **ตารางรวมต่อคลาส** ของทั้ง 2 โมเดล: AP50 / AP50-95 / P / R / F1 (จาก test) + ความถูกต้องต่อคลาส / % พลาดตรวจ / คลาสที่สับสนมากสุด (จาก confusion matrix) + แถวเฉลี่ยทุกคลาส → `figures/06_per_class_summary.png` + `reports/06_per_class_summary.csv`
2. **แกลเลอรี** แสดงรูปทั้งหมดใน `figures/` (01–06) ในหน้าเดียว + เขียน `reports/figure_gallery.html` (ฝังรูปในไฟล์เดียว เปิดได้ทุกที่ / แนบเล่มรายงานได้)
3. **ตัวอย่างภาพหลัง augmentation** (train_batch*.jpg ของทั้ง 2 experiment) + จำนวนภาพ-ครั้งที่โมเดลเห็น → `figures/06_augmentation_samples.png`

> รันได้ทุกครั้งที่อยากดูรูปรวม — ใช้เวลาไม่ถึง 1 นาที


In [ ]:
# === CELL 1 === โหลด config + ผลจาก nb06 (json/csv) — ไม่โหลดโมเดล ไม่ใช้ GPU
# ทำอะไร : อ่าน reports/06_evaluation.json, 06_per_class_ap_test.csv, 06_confusion_<model>.csv
# ทำไม   : ทุกตัวเลขในหน้านี้ต้องมาจากไฟล์ที่ nb06 บันทึกไว้ ไม่คำนวณใหม่ (ตัวเลขเดียวกับในเล่มรายงาน)
import sys, json, base64, importlib
from pathlib import Path
from datetime import datetime
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage, HTML

PROJECT_ROOT = Path.cwd().resolve()
CFG_PATH = PROJECT_ROOT / "configs" / "project.yaml"
with open(CFG_PATH, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)
DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
CLASSES = {int(k): v for k, v in CONFIG["classes"].items()}
NAMES = list(CLASSES.values())
NC = len(CLASSES)
sys.path.insert(0, str(DIRS["src"]))
import viz
importlib.reload(viz)
viz.apply_style()
EVAL = DIRS["reports"] / "06_evaluation.json"
assert EVAL.exists(), "ไม่พบ reports/06_evaluation.json → รัน 06_evaluate.ipynb ให้จบก่อน"
with open(EVAL, encoding="utf-8") as f:
    E = json.load(f)
FINAL = E["final_model"]["name"]
MODELS = list(E["test"])
per_class = pd.read_csv(DIRS["reports"] / "06_per_class_ap_test.csv")
cms = {m: pd.read_csv(DIRS["reports"] / f"06_confusion_{m}.csv", index_col=0) for m in MODELS}
print(f"[OK] final model = {FINAL} | models = {MODELS} | ไฟล์รูปใน figures/: {sum(1 for p in DIRS['figures'].iterdir() if p.suffix.lower() in ('.png', '.jpg'))} ไฟล์")


In [ ]:
# === CELL 2 === ตารางรวมต่อคลาส (test): AP50, AP50-95, P, R, F1 + จาก confusion matrix: ถูกต้อง %, พลาดตรวจ %, สับสนไปคลาสไหนมากสุด
# ทำอะไร : รวม 2 แหล่ง (per-class AP จาก val + confusion matrix ที่ conf 0.25) เป็นตารางเดียวต่อโมเดล + แถว "all (mean)" → heatmap + csv
# ทำไม   : ผู้อ่านรายงานอยากเห็น "คลาสไหนดี/แย่" ในตารางเดียว ไม่ต้องเปิดหลายรูป
LABELS = NAMES + ["background"]
rows = []
for m in MODELS:
    M = cms[m].to_numpy(dtype=float)            # [pred, true] (nc+1)²
    col = M.sum(axis=0)
    for j, cls in enumerate(NAMES):
        pc = per_class[(per_class.model == m) & (per_class["class"] == cls)].iloc[0]
        off = [(M[i, j] / max(col[j], 1), LABELS[i]) for i in range(NC + 1) if i != j]
        worst, worst_name = max(off)
        rows.append({"model": m, "class": cls, "AP50": pc["AP50"], "AP50-95": pc["AP50-95"], "P": pc["P"], "R": pc["R"], "F1": pc["F1"],
                     "correct_%": M[j, j] / max(col[j], 1) * 100, "missed_%": M[NC, j] / max(col[j], 1) * 100,
                     "top_confusion": f"{worst_name} {worst * 100:.1f}%", "n_true": int(col[j])})
    sub = pd.DataFrame([r for r in rows if r["model"] == m])
    rows.append({"model": m, "class": "all (mean)", **{k: float(sub[k].mean()) for k in ("AP50", "AP50-95", "P", "R", "F1", "correct_%", "missed_%")},
                 "top_confusion": "—", "n_true": int(sub["n_true"].sum())})
summary = pd.DataFrame(rows).set_index(["model", "class"])
summary.round(4).to_csv(DIRS["reports"] / "06_per_class_summary.csv", encoding="utf-8-sig")
pd.set_option("display.width", 200)
for m in MODELS:
    print(f"— {m}{' (FINAL)' if m == FINAL else ''} — per-class summary (test) —")
    print(summary.loc[m].round(3).to_string())
    print()

# --- heatmap: แถว = คลาส (+all), คอลัมน์ = metric, 1 panel ต่อโมเดล ---
metrics = ["AP50", "AP50-95", "P", "R", "F1"]
fig, axes = plt.subplots(1, len(MODELS), figsize=(6.2 * len(MODELS), 3.9))
axes = np.atleast_1d(axes)
for ax, m in zip(axes, MODELS):
    tab = summary.loc[m][metrics].astype(float)
    im = ax.imshow(tab.to_numpy(), cmap="Blues", vmin=0.5, vmax=1.0, aspect="auto")
    ax.set_xticks(range(len(metrics)), metrics)
    ax.set_yticks(range(len(tab)), [f"{c}  (n={int(summary.loc[(m, c), 'n_true'])})" for c in tab.index])
    ax.grid(False)
    for i in range(tab.shape[0]):
        for j in range(tab.shape[1]):
            v = tab.iloc[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=9, color="white" if v > 0.85 else viz.TEXT["primary"])
    extra = summary.loc[m]
    ax.set_title(f"{m}{' — FINAL' if m == FINAL else ''}  (correct@0.25: " + ", ".join(f"{c} {extra.loc[c, 'correct_%']:.0f}%" for c in NAMES) + ")", loc="left", fontsize=9.5)
fig.colorbar(im, ax=axes.tolist(), fraction=0.02, pad=0.02)
fig.suptitle("Per-class summary on the test set (AP from val, correct% from confusion matrix at conf 0.25)", x=0.01, ha="left", fontsize=11.5)
out = viz.save_fig(fig, DIRS["figures"], "06_per_class_summary.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)} | reports/06_per_class_summary.csv")


In [ ]:
# === CELL 3 === แกลเลอรีรูปทั้งหมด (inline) + เขียน reports/figure_gallery.html ที่ฝังรูปทุกรูปในไฟล์เดียว
# ทำอะไร : ไล่ figures/*.png เรียงตามชื่อ (01_… → 06_…) แสดงพร้อม caption; HTML ฝัง base64 → เปิดได้โดยไม่ต้องมีไฟล์รูปข้าง ๆ
# ทำไม   : ตอบคำถาม "ดูรูปทั้งหมดตอนนี้ได้ไหม" — และไฟล์ HTML เดียวส่งให้เพื่อน/อาจารย์ดูได้ทันที
CAPTIONS = {
    "01_smoke": "nb01 — smoke test ของ yolo26s.pt (COCO) บนภาพตัวอย่าง", "02_class_distribution": "nb02 — การกระจายคลาสหลังรวม 2 ชุดข้อมูล",
    "03_eda_overview": "nb03 — EDA: ขนาดภาพ / bbox ต่อภาพ / ขนาด bbox / ตำแหน่ง", "03_samples_by_class": "nb03 — ตัวอย่างภาพต่อคลาสต่อ source",
    "03_mendeley_name_vs_label": "nb03 — Mendeley: ชื่อไฟล์กับ label ไม่ตรงกัน", "03_blur_distribution": "nb03 — blur ratio + threshold",
    "03_blur_removed_examples": "nb03 — ภาพที่ถูกตัดเพราะเบลอ", "03_duplicate_distance": "nb03 — ระยะ pHash ใกล้สุดต่อภาพ",
    "03_duplicate_examples": "nb03 — คู่ภาพซ้ำ (เก็บ/ลบ)", "03_same_scene_diff_color_examples": "nb03 — ภาพเดิม สีต่าง label ต่าง (เก็บไว้ + flag)",
    "03_label_issues_examples": "nb03 — ปัญหา label (bbox จิ๋ว/ซ้ำ/ขัดแย้ง)", "03_exposure_distribution": "nb03 — ความสว่าง/saturation ของวัตถุ",
    "03_exposure_removed_examples": "nb03 — ภาพที่ถูกตัดเพราะแสง/ขนาด", "03_class_balance_before_after": "nb03 — class balance ก่อน/หลัง cleaning",
    "04_split_distribution": "nb04 — สัดส่วนคลาสต่อ split", "04_oversampling_effect": "nb04 — ผลของ oversampling ใน train",
    "04_occlusion_examples": "nb04 — ตัวอย่าง occlusion test subset", "05_training_curves_quick": "nb05 — learning curves (เบื้องต้น)",
    "06_per_class_ap_test": "nb06 — AP ต่อคลาส (test)", "06_confusion_matrix_baseline": "nb06 — confusion matrix: baseline",
    "06_confusion_matrix_improved": "nb06 — confusion matrix: improved", "06_learning_curves": "nb06 — learning curves + วินิจฉัย",
    "06_pr_f1_curves": "nb06 — PR curve + F1-confidence", "06_occlusion_nms": "nb06 — occlusion subset, sweep IoU/conf, NMS vs NMS-free",
    "06_robustness": "nb06 — robustness: mAP drop", "06_speed_benchmark": "nb06 — latency/FPS", "06_per_class_summary": "nb06b — ตารางรวมต่อคลาส",
    "06_augmentation_samples": "nb06b — ภาพ batch หลัง augmentation (Baseline vs Improved)",
}
figs = sorted(p for p in DIRS["figures"].iterdir() if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
groups = {}
for p in figs:
    groups.setdefault(p.name[:2], []).append(p)
print(f"[OK] พบ {len(figs)} รูป: " + ", ".join(f"nb{k} {len(v)}" for k, v in sorted(groups.items())))
for k, ps in sorted(groups.items()):
    display(HTML(f"<h3 style='margin:18px 0 4px'>Notebook {k}</h3>"))
    for p in ps:
        display(HTML(f"<div style='color:#52514e;font-size:13px;margin:8px 0 2px'><b>{p.name}</b> — {CAPTIONS.get(p.stem, CAPTIONS.get(p.stem[:8], ''))}</div>"))
        display(IPImage(filename=str(p), width=900))

# --- HTML ไฟล์เดียว ---
comp_md = DIRS["reports"] / "06_comparison_table.md"
parts = [f"<html><head><meta charset='utf-8'><title>Banana Ripeness YOLO26s — figure gallery</title>",
         "<style>body{font-family:system-ui,sans-serif;max-width:1100px;margin:24px auto;padding:0 16px;color:#0b0b0b}"
         "h1{font-size:22px}h2{font-size:17px;margin-top:28px;border-bottom:1px solid #e6e5e1}figure{margin:16px 0}img{max-width:100%;height:auto;border:1px solid #e6e5e1}"
         "figcaption{color:#52514e;font-size:13px;margin-top:4px}pre{background:#f6f6f4;padding:10px;font-size:12px;overflow:auto}</style></head><body>",
         f"<h1>Banana Ripeness Detection (YOLO26s) — figure gallery</h1><p>generated {datetime.now():%Y-%m-%d %H:%M} · final model = <b>{FINAL}</b> · {len(figs)} figures</p>"]
if comp_md.exists():
    parts.append("<h2>Baseline vs Improved (from reports/06_comparison_table.md)</h2><pre>" + comp_md.read_text(encoding="utf-8").replace("<", "&lt;") + "</pre>")
for k, ps in sorted(groups.items()):
    parts.append(f"<h2>Notebook {k}</h2>")
    for p in ps:
        b64 = base64.b64encode(p.read_bytes()).decode("ascii")
        parts.append(f"<figure><img src='data:image/{'png' if p.suffix.lower() == '.png' else 'jpeg'};base64,{b64}' alt='{p.name}'><figcaption><b>{p.name}</b> — {CAPTIONS.get(p.stem, CAPTIONS.get(p.stem[:8], ''))}</figcaption></figure>")
parts.append("</body></html>")
html_path = DIRS["reports"] / "figure_gallery.html"
html_path.write_text("\n".join(parts), encoding="utf-8")
print(f"[OK] {html_path.relative_to(PROJECT_ROOT)} ({html_path.stat().st_size / 1e6:.1f} MB) — ดับเบิลคลิกใน file browser ของ JupyterLab (กด Trust HTML) หรือดาวน์โหลดไปเปิดในเบราว์เซอร์")


In [ ]:
# === CELL 4 === ตัวอย่างภาพหลัง augmentation ของทั้ง 2 experiment + จำนวน "ภาพ-ครั้ง" ที่โมเดลเห็นจริง
# ทำอะไร : แสดง train_batch0-2.jpg ที่ ultralytics บันทึกตอนเริ่มเทรน (ภาพผ่าน mosaic/flip/HSV แล้ว พร้อม bbox) เรียง Baseline vs Improved
#          และคำนวณ ภาพต่อ epoch × epochs → บันทึกเป็น figures/06_augmentation_samples.png
# ทำไม   : augmentation เป็นแบบ online ไม่มีไฟล์ภาพเพิ่มบนดิสก์ — หลักฐานเดียวคือภาพ batch ที่บันทึกไว้ และตัวเลขภาพ-ครั้ง
from PIL import Image
aug_rows, tiles = [], {}
for name, rec in CONFIG["experiments"].items():
    run_dir = PROJECT_ROOT / rec["run_dir"]
    if name == "improved" and "train_list" in CONFIG.get("data_yaml", {}):
        n_per_epoch = sum(1 for l in open(PROJECT_ROOT / CONFIG["data_yaml"]["train_list"], encoding="utf-8") if l.strip())
    else:
        n_per_epoch = int(CONFIG["datasets"]["actual_split"]["train"])
    seen = n_per_epoch * int(rec["epochs_run"])
    aug_rows.append({"experiment": name, "images_per_epoch": n_per_epoch, "epochs": int(rec["epochs_run"]), "augmented_views_seen": seen,
                     "unique_files_on_disk": int(CONFIG["datasets"]["actual_split"]["train"])})
    tiles[name] = [p for p in (run_dir / f"train_batch{k}.jpg" for k in range(3)) if p.exists()]
    print(f"[{name}] ภาพต่อ epoch {n_per_epoch:,} × {rec['epochs_run']} epochs = {seen:,} ภาพ-ครั้ง (สุ่ม augmentation ใหม่ทุกครั้ง) | ไฟล์จริงบนดิสก์ {aug_rows[-1]['unique_files_on_disk']:,} | ตัวอย่าง batch: {len(tiles[name])} ไฟล์")
pd.DataFrame(aug_rows).to_csv(DIRS["reports"] / "06_augmentation_views.csv", index=False, encoding="utf-8-sig")
if any(tiles.values()):
    ncol = max(len(v) for v in tiles.values())
    fig, axes = plt.subplots(len(tiles), ncol, figsize=(5.2 * ncol, 5.2 * len(tiles)))
    axes = np.array(axes).reshape(len(tiles), ncol)
    for i, (name, ps) in enumerate(tiles.items()):
        for j in range(ncol):
            ax = axes[i, j]
            ax.axis("off")
            if j < len(ps):
                ax.imshow(Image.open(ps[j]).convert("RGB"))
                ax.set_title(f"{name} — {ps[j].name}", loc="left", fontsize=9)
    fig.suptitle("Augmented training batches as fed to the model (mosaic + flips + HSV) — Baseline (top) vs Improved (bottom)", x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    out = viz.save_fig(fig, DIRS["figures"], "06_augmentation_samples.png")
    plt.show()
    plt.close(fig)
    print(f"[OK] {out.relative_to(PROJECT_ROOT)} | reports/06_augmentation_views.csv")
else:
    print("[WARN] ไม่พบ train_batch*.jpg ใน runs/detect/<name>/ (เทรนด้วย plots=False?) — ตัวเลขภาพ-ครั้งยังถูกบันทึกใน reports/06_augmentation_views.csv")
